---
## Phase 03 — Graph Construction

**Goal:** Convert the clean flat dataframe from Phase 01 into a heterogeneous graph that PyTorch can work with.

A heterogeneous graph means we have **multiple node types** and **multiple edge types** — transaction nodes, card nodes, email nodes, device nodes — each with their own feature matrices. This preserves the full relational structure of the data.

By the end of this phase we will have:
- One feature matrix per node type
- One edge index tensor per edge type (12 total)
- Labels vector for transaction nodes
- Everything saved and ready for the GNN in Phase 04

### 3.1 — Imports and Load Data

We load the preprocessed dataframe from Phase 01 and the list of top 40 V-features we identified in Phase 02.

In [1]:
import pandas as pd
import numpy as np
import torch
import json
import os
import warnings
warnings.filterwarnings('ignore')

# Load preprocessed data from Phase 01
df = pd.read_csv('content/sample_data/df_preprocessed.csv')

# Load top 40 V-features identified in Phase 02
with open('content/sample_data/useful_v_features.json', 'r') as f:
    useful_v_features = json.load(f)

print(f'Loaded dataframe     : {df.shape[0]:,} rows x {df.shape[1]} cols')
print(f'Useful V-features    : {len(useful_v_features)}')
print(f'Fraud transactions   : {df["isFraud"].sum():,}  ({df["isFraud"].mean()*100:.2f}%)')

Loaded dataframe     : 590,540 rows x 434 cols
Useful V-features    : 40
Fraud transactions   : 20,663  (3.50%)


### 3.2 — Graph Schema Definition

Before writing any graph construction code, we formally define what our graph looks like.

**Node types:** Each unique value in each entity column becomes one node of that type.

| Node Type | Source Column | Approx. Count |
|---|---|---|
| Transaction | One per row | ~590k |
| Card1 | card1 | ~13,000 |
| Card2 | card2 | ~500 |
| Card3 | card3 | ~114 |
| Card4 | card4 | 4 |
| Card5 | card5 | ~119 |
| Card6 | card6 | 3 |
| AddrZip | addr1 | ~332 |
| AddrCountry | addr2 | ~75 |
| EmailPurchaser | P_emaildomain | ~59 |
| EmailRecipient | R_emaildomain | ~60 |
| DeviceType | DeviceType | 3 |
| DeviceInfo | DeviceInfo | ~1786 |

**Edge types:** One per entity column, always connecting a transaction node to an entity node.

```
Transaction → Card1, Card2, Card3, Card4, Card5, Card6
Transaction → AddrZip, AddrCountry
Transaction → EmailPurchaser, EmailRecipient
Transaction → DeviceType, DeviceInfo
```

**Node features:**
- Transaction nodes → top 40 V-features + TransactionAmt + TransactionDT (normalized)
- Entity nodes → degree vector (how many transactions each entity node is connected to)

In [2]:
# Entity columns and their node type names
ENTITY_COLS = {
    'card1':         'card1',
    'card2':         'card2',
    'card3':         'card3',
    'card4':         'card4',
    'card5':         'card5',
    'card6':         'card6',
    'addr1':         'addr1',
    'addr2':         'addr2',
    'P_emaildomain': 'email_p',
    'R_emaildomain': 'email_r',
    'DeviceType':    'device_type',
    'DeviceInfo':    'device_info',
}

# Minimum number of transactions an entity must appear in to get its own node
# Values below this threshold are grouped into a single 'rare' node
MIN_COUNT_THRESHOLD = 5

print('Graph schema defined.')
print(f'Entity columns      : {len(ENTITY_COLS)}')
print(f'Edge types          : {len(ENTITY_COLS)}')
print(f'Min count threshold : {MIN_COUNT_THRESHOLD}')

Graph schema defined.
Entity columns      : 12
Edge types          : 12
Min count threshold : 5


### 3.3 — Build Entity-to-Index Mappings

For each entity column, we create a dictionary that maps each unique value to a node index.

For example for `P_emaildomain`:
```
{'gmail.com': 0, 'yahoo.com': 1, 'hotmail.com': 2, ... , 'rare': 58}
```

Values that appear fewer than `MIN_COUNT_THRESHOLD` times get mapped to a special `'rare'` node instead of getting their own node. This prevents the graph from having thousands of near-empty nodes with no useful signal.

In [3]:
entity_mappings = {}   # col -> {value: node_index}
entity_num_nodes = {}  # col -> total number of nodes for this type

for col in ENTITY_COLS:
    # Count how many times each value appears
    value_counts = df[col].value_counts()

    # Keep only values that appear >= MIN_COUNT_THRESHOLD times
    frequent_values = value_counts[value_counts >= MIN_COUNT_THRESHOLD].index.tolist()

    # Assign an index to each frequent value starting from 0
    mapping = {val: idx for idx, val in enumerate(frequent_values)}

    # All rare values map to the last index — the 'rare' node
    rare_idx = len(frequent_values)
    mapping['__rare__'] = rare_idx

    entity_mappings[col] = mapping
    entity_num_nodes[col] = rare_idx + 1  # total nodes = frequent + 1 rare node

print(f'{"Column":<20} {"Frequent Nodes":>15} {"Rare→1 Node":>12} {"Total Nodes":>12}')
print('-' * 62)
for col in ENTITY_COLS:
    n_total    = entity_num_nodes[col]
    n_frequent = n_total - 1
    n_rare     = (df[col].value_counts() < MIN_COUNT_THRESHOLD).sum()
    print(f'{col:<20} {n_frequent:>15,} {n_rare:>12,} {n_total:>12,}')

Column                Frequent Nodes  Rare→1 Node  Total Nodes
--------------------------------------------------------------
card1                          6,512        7,041        6,513
card2                            500            0          501
card3                             76           38           77
card4                              5            0            6
card5                             76           43           77
card6                              5            0            6
addr1                            133          199          134
addr2                             30           44           31
P_emaildomain                     60            0           61
R_emaildomain                     61            0           62
DeviceType                         3            0            4
DeviceInfo                       888          899          889


### 3.4 — Build Transaction Node Features

Each transaction node gets a feature vector containing:
- The top 40 V-features identified in Phase 02
- `TransactionAmt` — the transaction amount
- `TransactionDT` — the transaction timestamp

This gives us a feature matrix of shape `(num_transactions, 42)`.

In [4]:
# Transaction node feature columns
transaction_feature_cols = useful_v_features + ['TransactionAmt', 'TransactionDT']

# Build the feature matrix — shape: (num_transactions, num_features)
transaction_features = df[transaction_feature_cols].values.astype(np.float32)

# Convert to PyTorch tensor
x_transaction = torch.tensor(transaction_features, dtype=torch.float)

# Labels — isFraud for each transaction node
y = torch.tensor(df['isFraud'].values, dtype=torch.long)

print(f'Transaction feature matrix : {x_transaction.shape}')
print(f'  → {x_transaction.shape[0]:,} transaction nodes')
print(f'  → {x_transaction.shape[1]} features per node')
print(f'Labels vector              : {y.shape}')
print(f'  → Fraud: {y.sum().item():,}  Legitimate: {(y==0).sum().item():,}')

Transaction feature matrix : torch.Size([590540, 42])
  → 590,540 transaction nodes
  → 42 features per node
Labels vector              : torch.Size([590540])
  → Fraud: 20,663  Legitimate: 569,877


### 3.5 — Build Edge Indices

For each entity column we build an **edge index tensor** of shape `(2, num_edges)`.

Row 0 = source node indices (transaction nodes)
Row 1 = destination node indices (entity nodes)

So if transaction 0 used card1 value 13926 which maps to entity node index 42:
```
edge_index_card1[:, 0] = [0, 42]
```

We make edges **bidirectional** — if a transaction connects to a card node, the card node also connects back to the transaction. This allows information to flow in both directions during message passing.

In [5]:
edge_indices = {}  # col -> edge_index tensor of shape (2, num_edges)

for col, node_type in ENTITY_COLS.items():
    mapping = entity_mappings[col]

    # Map each transaction's value to an entity node index
    # If value is rare (not in mapping), use the rare node index
    rare_idx = mapping['__rare__']
    entity_node_indices = df[col].apply(
        lambda v: mapping.get(v, rare_idx)
    ).values

    # Transaction node indices are just 0, 1, 2, ... num_transactions-1
    transaction_node_indices = np.arange(len(df))

    # Build bidirectional edge index
    # Forward: transaction -> entity
    src_fwd = torch.tensor(transaction_node_indices, dtype=torch.long)
    dst_fwd = torch.tensor(entity_node_indices,      dtype=torch.long)

    # Backward: entity -> transaction
    src_bwd = torch.tensor(entity_node_indices,      dtype=torch.long)
    dst_bwd = torch.tensor(transaction_node_indices, dtype=torch.long)

    # Stack forward and backward together
    src = torch.cat([src_fwd, src_bwd])
    dst = torch.cat([dst_fwd, dst_bwd])

    edge_indices[col] = torch.stack([src, dst], dim=0)

print(f'{"Entity Column":<20} {"Edge Index Shape":>20} {"Num Edges (one dir)":>22}')
print('-' * 65)
for col, ei in edge_indices.items():
    num_edges_one_dir = ei.shape[1] // 2
    print(f'{col:<20} {str(ei.shape):>20} {num_edges_one_dir:>22,}')

Entity Column            Edge Index Shape    Num Edges (one dir)
-----------------------------------------------------------------
card1                torch.Size([2, 1181080])                590,540
card2                torch.Size([2, 1181080])                590,540
card3                torch.Size([2, 1181080])                590,540
card4                torch.Size([2, 1181080])                590,540
card5                torch.Size([2, 1181080])                590,540
card6                torch.Size([2, 1181080])                590,540
addr1                torch.Size([2, 1181080])                590,540
addr2                torch.Size([2, 1181080])                590,540
P_emaildomain        torch.Size([2, 1181080])                590,540
R_emaildomain        torch.Size([2, 1181080])                590,540
DeviceType           torch.Size([2, 1181080])                590,540
DeviceInfo           torch.Size([2, 1181080])                590,540


### 3.6 — Build Entity Node Features

Entity nodes (card, email, device) don't have their own raw features the way transaction nodes do. Instead we give each entity node a feature vector based on:

- **Degree** — how many transactions this entity node is connected to
- **Fraud degree** — how many of those transactions are fraud
- **Fraud rate** — fraud degree / degree

These three numbers give each entity node meaningful context. A card node connected to 500 transactions with 200 fraud cases is very different from a card node connected to 500 transactions with 2 fraud cases.

In [6]:
entity_features = {}  # col -> feature tensor of shape (num_entity_nodes, 3)

for col, node_type in ENTITY_COLS.items():
    mapping  = entity_mappings[col]
    rare_idx = mapping['__rare__']
    n_nodes  = entity_num_nodes[col]

    # Map each transaction to its entity node index
    entity_node_indices = df[col].apply(
        lambda v: mapping.get(v, rare_idx)
    ).values

    # Initialize feature arrays
    degree       = np.zeros(n_nodes, dtype=np.float32)
    fraud_degree = np.zeros(n_nodes, dtype=np.float32)

    # Count degree and fraud degree for each entity node
    for txn_idx, entity_idx in enumerate(entity_node_indices):
        degree[entity_idx]       += 1
        fraud_degree[entity_idx] += df['isFraud'].iloc[txn_idx]

    # Fraud rate — avoid division by zero
    fraud_rate = np.where(degree > 0, fraud_degree / degree, 0.0).astype(np.float32)

    # Normalize degree to 0-1 range
    max_degree = degree.max() if degree.max() > 0 else 1.0
    degree_norm = (degree / max_degree).astype(np.float32)

    # Stack into feature matrix: (n_nodes, 3)
    feat_matrix = np.stack([degree_norm, fraud_degree / (fraud_degree.max() + 1e-8), fraud_rate], axis=1)
    entity_features[col] = torch.tensor(feat_matrix, dtype=torch.float)

print(f'{"Entity Column":<20} {"Feature Matrix Shape":>22}')
print('-' * 44)
for col, feat in entity_features.items():
    print(f'{col:<20} {str(feat.shape):>22}')

Entity Column          Feature Matrix Shape
--------------------------------------------
card1                 torch.Size([6513, 3])
card2                  torch.Size([501, 3])
card3                   torch.Size([77, 3])
card4                    torch.Size([6, 3])
card5                   torch.Size([77, 3])
card6                    torch.Size([6, 3])
addr1                  torch.Size([134, 3])
addr2                   torch.Size([31, 3])
P_emaildomain           torch.Size([61, 3])
R_emaildomain           torch.Size([62, 3])
DeviceType               torch.Size([4, 3])
DeviceInfo             torch.Size([889, 3])


### 3.7 — Package the Heterogeneous Graph

We package everything into a clean Python dictionary that represents the full heterogeneous graph. This is what Phase 04 will load directly.

Structure:
```
graph = {
    'node_features': {
        'transaction': x_transaction,   # (590k, 42)
        'card1':       x_card1,         # (13001, 3)
        'email_p':     x_email_p,       # (60, 3)
        ...                             # one per entity type
    },
    'edge_indices': {
        'card1':       edge_index_card1, # (2, 2*590k)
        'email_p':     edge_index_email, # (2, 2*590k)
        ...                              # one per entity column
    },
    'labels': y,                         # (590k,)
    'num_nodes': {...},                  # node count per type
    'metadata': {...}                    # schema info
}
```

In [7]:
# Build node features dict — transaction + all entity types
node_features = {'transaction': x_transaction}
for col, node_type in ENTITY_COLS.items():
    node_features[node_type] = entity_features[col]

# Build num_nodes dict
num_nodes = {'transaction': len(df)}
for col, node_type in ENTITY_COLS.items():
    num_nodes[node_type] = entity_num_nodes[col]

# Package full graph
graph = {
    'node_features': node_features,
    'edge_indices':  edge_indices,
    'labels':        y,
    'num_nodes':     num_nodes,
    'metadata': {
        'entity_cols':         list(ENTITY_COLS.keys()),
        'node_types':          list(ENTITY_COLS.values()),
        'transaction_features': transaction_feature_cols,
        'num_transaction_features': len(transaction_feature_cols),
        'min_count_threshold': MIN_COUNT_THRESHOLD,
    }
}

print('Heterogeneous graph packaged.')
print()
print('Node types and sizes:')
for ntype, n in graph['num_nodes'].items():
    feat_dim = graph['node_features'][ntype].shape[1]
    print(f'  {ntype:<20} {n:>10,} nodes  |  {feat_dim} features')
print()
print('Edge types and sizes:')
for col, ei in graph['edge_indices'].items():
    print(f'  transaction → {col:<15} {ei.shape[1]//2:>10,} edges (bidirectional)')

Heterogeneous graph packaged.

Node types and sizes:
  transaction             590,540 nodes  |  42 features
  card1                     6,513 nodes  |  3 features
  card2                       501 nodes  |  3 features
  card3                        77 nodes  |  3 features
  card4                         6 nodes  |  3 features
  card5                        77 nodes  |  3 features
  card6                         6 nodes  |  3 features
  addr1                       134 nodes  |  3 features
  addr2                        31 nodes  |  3 features
  email_p                      61 nodes  |  3 features
  email_r                      62 nodes  |  3 features
  device_type                   4 nodes  |  3 features
  device_info                 889 nodes  |  3 features

Edge types and sizes:
  transaction → card1              590,540 edges (bidirectional)
  transaction → card2              590,540 edges (bidirectional)
  transaction → card3              590,540 edges (bidirectional)
  transaction

### 3.8 — Validation

Before saving, we run sanity checks to confirm the graph is structurally correct.

In [8]:
checks = {}

# 1. Transaction feature matrix has correct shape
checks['Transaction features shape correct'] = (
    graph['node_features']['transaction'].shape == (len(df), len(transaction_feature_cols))
)

# 2. Labels length matches transaction count
checks['Labels length matches transactions'] = (
    len(graph['labels']) == len(df)
)

# 3. Each edge index has correct number of edges (bidirectional = 2 * num_transactions)
checks['Edge indices are bidirectional'] = all(
    ei.shape[1] == 2 * len(df)
    for ei in graph['edge_indices'].values()
)

# 4. No negative node indices in any edge index
checks['No negative node indices'] = all(
    ei.min().item() >= 0
    for ei in graph['edge_indices'].values()
)

# 5. Entity node indices within valid range
checks['Entity node indices in valid range'] = all(
    graph['edge_indices'][col][1, :len(df)].max().item() < entity_num_nodes[col]
    for col in ENTITY_COLS
)

# 6. No NaN values in any feature matrix
checks['No NaN in feature matrices'] = all(
    not torch.isnan(feat).any().item()
    for feat in graph['node_features'].values()
)

# 7. Labels are only 0 and 1
checks['Labels are binary'] = (
    set(graph['labels'].unique().tolist()) == {0, 1}
)

all_pass = True
for name, result in checks.items():
    status = '✓' if result else '✗'
    print(f'  {status}  {name}')
    if not result:
        all_pass = False

print()
if all_pass:
    print('All checks passed — graph is valid.')
else:
    print('Some checks failed — fix before saving.')

  ✓  Transaction features shape correct
  ✓  Labels length matches transactions
  ✓  Edge indices are bidirectional
  ✓  No negative node indices
  ✓  Entity node indices in valid range
  ✓  No NaN in feature matrices
  ✓  Labels are binary

All checks passed — graph is valid.


### 3.9 — Save the Graph

We save the full graph using `torch.save()`. This preserves all tensor types and shapes exactly. Phase 04 will load this file directly with `torch.load()`.

In [9]:
if all_pass:
    save_path = 'content/sample_data/hetero_graph.pt'
    torch.save(graph, save_path)
    file_size_mb = os.path.getsize(save_path) / (1024 * 1024)
    print(f'Graph saved → hetero_graph.pt  ({file_size_mb:.1f} MB)')
    print()
    print('Summary:')
    print(f'  Node types  : {len(graph["num_nodes"])}')
    print(f'  Edge types  : {len(graph["edge_indices"])}')
    total_nodes = sum(graph['num_nodes'].values())
    total_edges = sum(ei.shape[1] for ei in graph['edge_indices'].values())
    print(f'  Total nodes : {total_nodes:,}')
    print(f'  Total edges : {total_edges:,}')
    print()
    print('Phase 03 complete. Next → Phase 04: GNN Model.')
else:
    print('Graph not saved — fix validation errors first.')

Graph saved → hetero_graph.pt  (315.5 MB)

Summary:
  Node types  : 13
  Edge types  : 12
  Total nodes : 598,901
  Total edges : 14,172,960

Phase 03 complete. Next → Phase 04: GNN Model.
